# Phase 2 — Feature Engineering and Training Dataset Generation

- **2A.** Initialize and inspect the feature notebook
- **2B.** Create reference-time PM2.5 features
- **2C.** Current weather, time, and wind-direction features
- **2D.** Expand reference rows into horizons 1–72
- **2E.** Attach target PM2.5 and target-hour weather
- **2F.** Remove invalid rows and run leakage checks
- **2G.** Chronological train/validation/test split
- **2H.** Save and validate the datasets

## **2A.** Prepare notebook and raw filesLoad and validate the canonical hourly dataset

The canonical dataset created in Phase 1 contains one row for every UTC hour
in the validated historical period.

Before creating lag, rolling, weather, time, and forecast-horizon features,
this notebook first verifies that the input dataset still satisfies the
assumptions required by the training pipeline.

This step checks:

- the canonical Parquet file can be loaded successfully
- timestamps are timezone-aware UTC
- rows are arranged chronologically
- there is exactly one row per expected hour
- duplicate timestamps are absent
- weather variables remain complete
- missing PM2.5 values match the previously validated sensor gaps

No model features are created in this subphase.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

CANONICAL_DATASET_PATH = (
    PROCESSED_DATA_DIR
    / "pearls_aqi_canonical_hourly.parquet"
)


print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Canonical dataset path:", CANONICAL_DATASET_PATH)
print("Canonical dataset exists:", CANONICAL_DATASET_PATH.exists())

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor
Processed data directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/processed
Canonical dataset path: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/processed/pearls_aqi_canonical_hourly.parquet
Canonical dataset exists: True


### Load the canonical hourly dataset

The Parquet file is the model-ready output of Phase 1.

At this stage, the dataset is still an hourly time series. It has not yet been
expanded into forecast horizons and it does not yet contain lag or rolling
features.

The original file is loaded into a new DataFrame so that the saved Phase 1
output remains unchanged.

In [4]:
if not CANONICAL_DATASET_PATH.exists():
    raise FileNotFoundError(
        "Canonical dataset was not found at:\n"
        f"{CANONICAL_DATASET_PATH}\n\n"
        "Run the Phase 1 canonical dataset notebook first."
    )


canonical_df = pd.read_parquet(
    CANONICAL_DATASET_PATH
)


print("Canonical dataset loaded successfully.")
print("Shape:", canonical_df.shape)

Canonical dataset loaded successfully.
Shape: (9144, 19)


### Inspect the dataset structure

Before writing feature-engineering code, the actual schema must be inspected
rather than assumed.

The following checks show:

- the complete column list
- the stored data type of each column
- sample rows from the beginning and end of the historical period

This helps detect accidental schema changes before feature generation begins.

In [5]:
print("Canonical columns:")

for index, column in enumerate(
    canonical_df.columns,
    start=1,
):
    print(f"{index:02d}. {column}")


print("\nData types:")

display(
    canonical_df.dtypes
    .astype(str)
    .rename("dtype")
    .to_frame()
)

Canonical columns:
01. datetime_utc
02. sensor_id
03. location_id
04. location_name
05. provider
06. pm25_ug_m3
07. pm25_missing
08. pm25_zero_flag
09. pm25_negative_flag
10. temperature_2m
11. relative_humidity_2m
12. dew_point_2m
13. surface_pressure
14. precipitation
15. rain
16. cloud_cover
17. wind_speed_10m
18. wind_direction_10m
19. wind_gusts_10m

Data types:


,dtype
datetime_utc,"datetime64[us, UTC]"
sensor_id,int64
location_id,int64
location_name,str
provider,str
pm25_ug_m3,float64
pm25_missing,bool
pm25_zero_flag,bool
pm25_negative_flag,bool
temperature_2m,float64


In [6]:
print("First five rows:")
display(canonical_df.head())


print("Last five rows:")
display(canonical_df.tail())

First five rows:


,datetime_utc,sensor_id,location_id,location_name,provider,pm25_ug_m3,pm25_missing,pm25_zero_flag,pm25_negative_flag,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2025-07-08 00:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,13.202458,False,False,False,28.5,87,26.2,997.3,0.1,0.1,49,3.6,225,8.6
1,2025-07-08 01:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,14.268958,False,False,False,29.1,82,25.8,997.9,0.0,0.0,98,1.5,263,5.4
2,2025-07-08 02:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,15.587583,False,False,False,29.4,80,25.6,998.5,0.0,0.0,82,0.7,270,5.0
3,2025-07-08 03:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,14.713375,False,False,False,30.2,75,25.3,998.9,0.0,0.0,79,2.2,228,10.1
4,2025-07-08 04:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,17.668875,False,False,False,31.1,70,24.9,999.3,0.0,0.0,32,3.2,199,13.7


Last five rows:


,datetime_utc,sensor_id,location_id,location_name,provider,pm25_ug_m3,pm25_missing,pm25_zero_flag,pm25_negative_flag,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
9139,2026-07-23 19:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,11.1,False,False,False,29.2,82,25.8,998.8,0.0,0.0,41,10.4,238,27.7
9140,2026-07-23 20:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,9.9,False,False,False,28.9,83,25.7,998.6,0.0,0.0,43,8.8,242,25.2
9141,2026-07-23 21:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,9.2,False,False,False,28.7,83,25.6,997.9,0.0,0.0,88,8.1,245,21.2
9142,2026-07-23 22:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,11.7,False,False,False,28.7,82,25.4,997.8,0.0,0.0,98,7.0,249,20.2
9143,2026-07-23 23:00:00+00:00,13387396,4814327,Zafar Memon DHA,AirGradient,8.3,False,False,False,28.6,83,25.4,997.7,0.0,0.0,45,5.2,254,16.9


### Define the expected input schema

Feature engineering should fail early when an important input column is
missing.

The required schema contains:

- one UTC timestamp
- the cleaned PM2.5 measurement
- PM2.5 quality flags
- static sensor and location metadata
- the ten selected historical weather variables

The weather variables are intentionally the same variables that will later be
requested from the Open-Meteo Forecast API during live prediction.

In [7]:
WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "surface_pressure",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
]


REQUIRED_CANONICAL_COLUMNS = [
    "datetime_utc",
    "sensor_id",
    "location_id",
    "location_name",
    "provider",
    "pm25_ug_m3",
    "pm25_missing",
    "pm25_zero_flag",
    "pm25_negative_flag",
    *WEATHER_COLUMNS,
]


missing_required_columns = [
    column
    for column in REQUIRED_CANONICAL_COLUMNS
    if column not in canonical_df.columns
]

unexpected_columns = [
    column
    for column in canonical_df.columns
    if column not in REQUIRED_CANONICAL_COLUMNS
]


print("Missing required columns:", missing_required_columns)
print("Unexpected columns:", unexpected_columns)

Missing required columns: []
Unexpected columns: []


In [8]:
assert not missing_required_columns, (
    "The canonical dataset is missing required columns: "
    f"{missing_required_columns}"
)

print("Required canonical schema validation passed.")

Required canonical schema validation passed.


### Validate the hourly UTC timeline

Time-series feature engineering depends on correct timestamp ordering.

Lag and rolling calculations only have the intended meaning when:

- timestamps are timezone-aware UTC
- rows are sorted from oldest to newest
- each timestamp appears once
- the expected historical range is preserved
- the dataset contains exactly one row for every expected hour

These checks protect the later feature calculations from subtle time-related
errors and data leakage.

In [9]:
EXPECTED_START_UTC = pd.Timestamp(
    "2025-07-08 00:00:00",
    tz="UTC",
)

EXPECTED_END_UTC = pd.Timestamp(
    "2026-07-23 23:00:00",
    tz="UTC",
)

EXPECTED_TIMELINE = pd.date_range(
    start=EXPECTED_START_UTC,
    end=EXPECTED_END_UTC,
    freq="h",
)


print("Expected start:", EXPECTED_START_UTC)
print("Expected end:", EXPECTED_END_UTC)
print("Expected hourly rows:", len(EXPECTED_TIMELINE))

Expected start: 2025-07-08 00:00:00+00:00
Expected end: 2026-07-23 23:00:00+00:00
Expected hourly rows: 9144


In [10]:
print(
    "Timestamp dtype:",
    canonical_df["datetime_utc"].dtype,
)

print(
    "First timestamp:",
    canonical_df["datetime_utc"].min(),
)

print(
    "Last timestamp:",
    canonical_df["datetime_utc"].max(),
)

print(
    "Chronologically sorted:",
    canonical_df["datetime_utc"]
    .is_monotonic_increasing,
)

print(
    "Duplicate timestamps:",
    canonical_df["datetime_utc"]
    .duplicated()
    .sum(),
)

Timestamp dtype: datetime64[us, UTC]
First timestamp: 2025-07-08 00:00:00+00:00
Last timestamp: 2026-07-23 23:00:00+00:00
Chronologically sorted: True
Duplicate timestamps: 0


In [11]:
actual_timestamps = pd.DatetimeIndex(
    canonical_df["datetime_utc"]
)

missing_timestamps = EXPECTED_TIMELINE.difference(
    actual_timestamps
)

unexpected_timestamps = actual_timestamps.difference(
    EXPECTED_TIMELINE
)


print("Actual rows:", len(canonical_df))
print("Missing expected timestamps:", len(missing_timestamps))
print("Unexpected timestamps:", len(unexpected_timestamps))

Actual rows: 9144
Missing expected timestamps: 0
Unexpected timestamps: 0


In [12]:
assert len(canonical_df) == len(EXPECTED_TIMELINE)

assert (
    canonical_df["datetime_utc"]
    .duplicated()
    .sum()
    == 0
)

assert canonical_df[
    "datetime_utc"
].is_monotonic_increasing

assert (
    canonical_df["datetime_utc"].min()
    == EXPECTED_START_UTC
)

assert (
    canonical_df["datetime_utc"].max()
    == EXPECTED_END_UTC
)

assert len(missing_timestamps) == 0
assert len(unexpected_timestamps) == 0

assert str(
    canonical_df["datetime_utc"].dt.tz
) == "UTC"


print("Canonical UTC timeline validation passed.")

Canonical UTC timeline validation passed.


### Inspect measurement completeness

The historical weather data was previously validated as complete.

PM2.5 contains known missing periods caused by sensor outages and five exact
zero readings that were treated as invalid measurements.

These missing PM2.5 values must remain missing. They will later prevent the
creation of training rows whose required historical or target PM2.5 values are
unavailable.

No PM2.5 interpolation is performed.

In [13]:
missing_summary = (
    canonical_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

display(missing_summary)

,missing_count
pm25_ug_m3,454
sensor_id,0
datetime_utc,0
location_id,0
location_name,0
provider,0
pm25_missing,0
pm25_zero_flag,0
pm25_negative_flag,0
temperature_2m,0


In [14]:
pm25_available_hours = int(
    canonical_df["pm25_ug_m3"]
    .notna()
    .sum()
)

pm25_missing_hours = int(
    canonical_df["pm25_ug_m3"]
    .isna()
    .sum()
)

pm25_coverage_percent = (
    pm25_available_hours
    / len(canonical_df)
    * 100
)

total_missing_weather_values = int(
    canonical_df[WEATHER_COLUMNS]
    .isna()
    .sum()
    .sum()
)


print("Available PM2.5 hours:", pm25_available_hours)
print("Missing PM2.5 hours:", pm25_missing_hours)

print(
    f"Cleaned PM2.5 coverage: "
    f"{pm25_coverage_percent:.2f}%"
)

print(
    "Total missing weather values:",
    total_missing_weather_values,
)

print(
    "Zero readings flagged:",
    canonical_df["pm25_zero_flag"].sum(),
)

print(
    "Negative readings flagged:",
    canonical_df["pm25_negative_flag"].sum(),
)

Available PM2.5 hours: 8690
Missing PM2.5 hours: 454
Cleaned PM2.5 coverage: 95.03%
Total missing weather values: 0
Zero readings flagged: 5
Negative readings flagged: 0


In [15]:
assert (
    canonical_df["pm25_ug_m3"]
    .isna()
    .sum()
    == 454
)

assert (
    canonical_df["pm25_missing"]
    .sum()
    == 454
)

assert (
    canonical_df["pm25_ug_m3"]
    .eq(0)
    .sum()
    == 0
)

assert (
    canonical_df["pm25_ug_m3"]
    .lt(0)
    .sum()
    == 0
)

assert (
    canonical_df["pm25_zero_flag"]
    .sum()
    == 5
)

assert (
    canonical_df["pm25_negative_flag"]
    .sum()
    == 0
)

assert total_missing_weather_values == 0

assert canonical_df[
    "pm25_missing"
].equals(
    canonical_df["pm25_ug_m3"].isna()
)


print("Phase 2A input dataset validation passed.")

Phase 2A input dataset validation passed.


## **2B.** Reference-time PM2.5 features

### PM2.5 current and lag values

Air pollution usually has temporal continuity. The PM2.5 concentration at a
given hour is often related to concentrations observed during the previous
several hours.

Lag features give the model a snapshot of this recent history.

For example:

- `pm25_lag_1h` represents the PM2.5 value one hour before the reference time
- `pm25_lag_6h` represents the value six hours before the reference time
- `pm25_lag_24h` represents the value at the same hour on the previous day

Only current and past PM2.5 measurements are used. No PM2.5 value after the
reference timestamp is included, which prevents target leakage.

Missing PM2.5 measurements remain missing. They are not interpolated.

In [16]:
feature_base_df = canonical_df.copy()

print("Feature base shape:", feature_base_df.shape)

Feature base shape: (9144, 19)


### Selected PM2.5 history periods

The initial feature set uses a small group of practical lag periods:

- 1 hour for immediate pollution conditions
- 3 hours for short-term movement
- 6 hours for recent intra-day conditions
- 12 hours for half-day context
- 24 hours for daily temporal patterns

This first version intentionally avoids creating a very large number of lag
features.

In [17]:
PM25_LAG_HOURS = [
    1,
    3,
    6,
    12,
    24,
]

print("PM2.5 lag periods:", PM25_LAG_HOURS)

PM2.5 lag periods: [1, 3, 6, 12, 24]


In [18]:
feature_base_df["pm25_current"] = (
    feature_base_df["pm25_ug_m3"]
)

for lag_hours in PM25_LAG_HOURS:
    feature_base_df[f"pm25_lag_{lag_hours}h"] = (
        feature_base_df["pm25_ug_m3"]
        .shift(lag_hours)
    )

### Inspect the generated lag values

The first rows naturally contain missing lag values because no earlier history
exists before the start of the dataset.

For example:

- the first row cannot have a one-hour lag
- the first three rows cannot have a complete three-hour lag history
- the first 24 rows cannot all have a 24-hour lag

These are expected boundary effects and will be handled later when valid
training rows are selected.

In [19]:
PM25_LAG_COLUMNS = [
    "pm25_current",
    "pm25_lag_1h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "pm25_lag_12h",
    "pm25_lag_24h",
]

display(
    feature_base_df[
        [
            "datetime_utc",
            *PM25_LAG_COLUMNS,
        ]
    ].head(30)
)

,datetime_utc,pm25_current,pm25_lag_1h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h
0,2025-07-08 00:00:00+00:00,13.202458,NaN,NaN,NaN,NaN,NaN
1,2025-07-08 01:00:00+00:00,14.268958,13.202458,NaN,NaN,NaN,NaN
2,2025-07-08 02:00:00+00:00,15.587583,14.268958,NaN,NaN,NaN,NaN
3,2025-07-08 03:00:00+00:00,14.713375,15.587583,13.202458,NaN,NaN,NaN
4,2025-07-08 04:00:00+00:00,17.668875,14.713375,14.268958,NaN,NaN,NaN
5,2025-07-08 05:00:00+00:00,16.109417,17.668875,15.587583,NaN,NaN,NaN
6,2025-07-08 06:00:00+00:00,15.611542,16.109417,14.713375,13.202458,NaN,NaN
7,2025-07-08 07:00:00+00:00,15.249333,15.611542,17.668875,14.268958,NaN,NaN
8,2025-07-08 08:00:00+00:00,14.307417,15.249333,16.109417,15.587583,NaN,NaN
9,2025-07-08 09:00:00+00:00,15.640125,14.307417,15.611542,14.713375,NaN,NaN


In [20]:
lag_missing_summary = (
    feature_base_df[PM25_LAG_COLUMNS]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(lag_missing_summary)

,missing_count
pm25_current,454
pm25_lag_1h,455
pm25_lag_3h,457
pm25_lag_6h,460
pm25_lag_12h,466
pm25_lag_24h,478


### Verify that lag values use only earlier timestamps

A few rows are checked manually by comparing each generated lag with the
original PM2.5 series at the corresponding earlier row.

This ensures that the lag direction is correct. Accidentally shifting in the
opposite direction would introduce future PM2.5 values and cause data leakage.

In [21]:
validation_row_indices = [
    24,
    100,
    1000,
    5000,
]

lag_validation_rows = []

for row_index in validation_row_indices:
    record = {
        "row_index": row_index,
        "reference_time": feature_base_df.loc[
            row_index,
            "datetime_utc",
        ],
        "pm25_current": feature_base_df.loc[
            row_index,
            "pm25_current",
        ],
    }

    for lag_hours in PM25_LAG_HOURS:
        record[f"generated_lag_{lag_hours}h"] = (
            feature_base_df.loc[
                row_index,
                f"pm25_lag_{lag_hours}h",
            ]
        )

        record[f"expected_lag_{lag_hours}h"] = (
            feature_base_df.loc[
                row_index - lag_hours,
                "pm25_ug_m3",
            ]
        )

    lag_validation_rows.append(record)


lag_validation_df = pd.DataFrame(
    lag_validation_rows
)

display(lag_validation_df)

,row_index,reference_time,pm25_current,generated_lag_1h,expected_lag_1h,generated_lag_3h,expected_lag_3h,generated_lag_6h,expected_lag_6h,generated_lag_12h,expected_lag_12h,generated_lag_24h,expected_lag_24h
0,24,2025-07-09 00:00:00+00:00,17.257208,15.527958,15.527958,16.360917,16.360917,19.673788,19.673788,19.074875,19.074875,13.202458,13.202458
1,100,2025-07-12 04:00:00+00:00,6.743500,6.556208,6.556208,6.987083,6.987083,6.636958,6.636958,9.951542,9.951542,12.865000,12.865000
2,1000,2025-08-18 16:00:00+00:00,32.246417,29.688917,29.688917,11.824041,11.824041,26.788500,26.788500,29.549042,29.549042,8.804500,8.804500
3,5000,2026-02-01 08:00:00+00:00,69.747167,71.419333,71.419333,114.152500,114.152500,124.377833,124.377833,150.486000,150.486000,55.602917,55.602917


In [22]:
for lag_hours in PM25_LAG_HOURS:
    generated_lag = feature_base_df[
        f"pm25_lag_{lag_hours}h"
    ]

    expected_lag = feature_base_df[
        "pm25_ug_m3"
    ].shift(lag_hours)

    pd.testing.assert_series_equal(
        generated_lag,
        expected_lag,
        check_names=False,
    )


assert feature_base_df[
    "pm25_current"
].equals(
    feature_base_df["pm25_ug_m3"]
)


print("PM2.5 current and lag feature validation passed.")

PM2.5 current and lag feature validation passed.


### Validate lag behavior around a PM2.5 outage

The canonical timeline contains every expected UTC hour, including hours when
the PM2.5 sensor did not report a measurement.

Because missing PM2.5 values were intentionally not interpolated, lag features
should also become missing whenever they point to an unavailable historical
measurement.

This check verifies that the feature pipeline does not accidentally bridge
sensor outages or treat non-consecutive observations as consecutive hourly
measurements.

In [23]:
missing_pm25_mask = feature_base_df[
    "pm25_ug_m3"
].isna()

missing_group_ids = missing_pm25_mask.ne(
    missing_pm25_mask.shift()
).cumsum()

missing_gap_summary = (
    feature_base_df.loc[
        missing_pm25_mask,
        ["datetime_utc"],
    ]
    .assign(
        gap_group=missing_group_ids[
            missing_pm25_mask
        ].to_numpy()
    )
    .groupby("gap_group")
    .agg(
        gap_start_utc=("datetime_utc", "min"),
        gap_end_utc=("datetime_utc", "max"),
        missing_hours=("datetime_utc", "size"),
    )
    .reset_index(drop=True)
    .sort_values(
        ["missing_hours", "gap_start_utc"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

display(missing_gap_summary.head(10))

,gap_start_utc,gap_end_utc,missing_hours
0,2025-07-28 08:00:00+00:00,2025-07-28 08:00:00+00:00,1
1,2025-09-29 11:00:00+00:00,2025-09-29 11:00:00+00:00,1
2,2025-09-30 14:00:00+00:00,2025-09-30 14:00:00+00:00,1
3,2025-10-04 06:00:00+00:00,2025-10-04 06:00:00+00:00,1
4,2025-10-11 11:00:00+00:00,2025-10-11 11:00:00+00:00,1
5,2025-10-25 22:00:00+00:00,2025-10-25 22:00:00+00:00,1
6,2025-11-05 06:00:00+00:00,2025-11-05 06:00:00+00:00,1
7,2025-11-13 05:00:00+00:00,2025-11-13 05:00:00+00:00,1
8,2025-11-14 01:00:00+00:00,2025-11-14 01:00:00+00:00,1
9,2025-11-16 03:00:00+00:00,2025-11-16 03:00:00+00:00,1


### Select one outage period

A short outage is selected so the values immediately before, during, and after
the missing period can be reviewed.

The inspection window includes 24 hours before and 30 hours after the gap.
This is long enough to observe how the 1-hour, 3-hour, 6-hour, 12-hour, and
24-hour lag features respond.

In [24]:
selected_gap = missing_gap_summary.iloc[0]

selected_gap_start = selected_gap["gap_start_utc"]
selected_gap_end = selected_gap["gap_end_utc"]

print("Selected gap start:", selected_gap_start)
print("Selected gap end:", selected_gap_end)
print(
    "Selected gap length:",
    selected_gap["missing_hours"],
    "hours",
)

Selected gap start: 2025-07-28 08:00:00+00:00
Selected gap end: 2025-07-28 08:00:00+00:00
Selected gap length: 1 hours


In [25]:
inspection_start = (
    selected_gap_start
    - pd.Timedelta(hours=24)
)

inspection_end = (
    selected_gap_end
    + pd.Timedelta(hours=30)
)

gap_lag_inspection = feature_base_df.loc[
    feature_base_df["datetime_utc"].between(
        inspection_start,
        inspection_end,
    ),
    [
        "datetime_utc",
        "pm25_current",
        "pm25_lag_1h",
        "pm25_lag_3h",
        "pm25_lag_6h",
        "pm25_lag_12h",
        "pm25_lag_24h",
    ],
].copy()

display(gap_lag_inspection)

,datetime_utc,pm25_current,pm25_lag_1h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h
464,2025-07-27 08:00:00+00:00,10.552375,13.741792,13.083958,19.169583,14.616375,14.987569
465,2025-07-27 09:00:00+00:00,10.303875,10.552375,12.455917,19.376000,14.986958,11.847583
466,2025-07-27 10:00:00+00:00,10.469125,10.303875,13.741792,16.613292,12.780334,10.010792
467,2025-07-27 11:00:00+00:00,11.636375,10.469125,10.552375,13.083958,10.994083,11.360500
468,2025-07-27 12:00:00+00:00,13.448292,11.636375,10.303875,12.455917,14.181708,17.057625
469,2025-07-27 13:00:00+00:00,12.918500,13.448292,10.469125,13.741792,15.088417,14.146625
470,2025-07-27 14:00:00+00:00,13.318834,12.918500,11.636375,10.552375,19.169583,14.053417
471,2025-07-27 15:00:00+00:00,14.581750,13.318834,13.448292,10.303875,19.376000,13.650333
472,2025-07-27 16:00:00+00:00,14.265625,14.581750,12.918500,10.469125,16.613292,12.662000
473,2025-07-27 17:00:00+00:00,12.010458,14.265625,13.318834,11.636375,13.083958,15.644125


### Verify that missing observations propagate to the correct lag timestamps

For every missing PM2.5 timestamp, a lag feature should be missing exactly
`lag_hours` later.

For example, a missing measurement at 10:00 UTC must make `pm25_lag_6h`
missing at 16:00 UTC.

This check validates the timestamp relationship directly rather than relying
only on visual inspection.

In [26]:
missing_pm25_timestamps = set(
    feature_base_df.loc[
        feature_base_df["pm25_ug_m3"].isna(),
        "datetime_utc",
    ]
)

for lag_hours in PM25_LAG_HOURS:
    expected_missing_lag_timestamps = {
        timestamp + pd.Timedelta(hours=lag_hours)
        for timestamp in missing_pm25_timestamps
    }

    expected_missing_lag_timestamps = (
        expected_missing_lag_timestamps
        .intersection(
            set(feature_base_df["datetime_utc"])
        )
    )

    actual_missing_lag_timestamps = set(
        feature_base_df.loc[
            feature_base_df[
                f"pm25_lag_{lag_hours}h"
            ].isna(),
            "datetime_utc",
        ]
    )

    boundary_timestamps = set(
        feature_base_df[
            "datetime_utc"
        ].iloc[:lag_hours]
    )

    expected_all_missing_timestamps = (
        expected_missing_lag_timestamps
        | boundary_timestamps
    )

    assert (
        actual_missing_lag_timestamps
        == expected_all_missing_timestamps
    ), (
        f"Missing-value propagation failed for "
        f"pm25_lag_{lag_hours}h"
    )

    print(
        f"pm25_lag_{lag_hours}h validation passed: "
        f"{len(actual_missing_lag_timestamps)} missing values"
    )

pm25_lag_1h validation passed: 455 missing values
pm25_lag_3h validation passed: 457 missing values
pm25_lag_6h validation passed: 460 missing values
pm25_lag_12h validation passed: 466 missing values
pm25_lag_24h validation passed: 478 missing values


### Selected rolling windows

The selected windows provide different levels of recent pollution context:

- 3 hours captures very recent conditions
- 6 hours summarizes short-term movement
- 12 hours captures half-day behavior
- 24 hours represents the recent daily pollution level

A complete window is required. Missing PM2.5 readings are not ignored when
calculating these features.

In [27]:
PM25_ROLLING_WINDOWS = [
    3,
    6,
    12,
    24,
]

print(
    "PM2.5 rolling windows:",
    PM25_ROLLING_WINDOWS,
)

PM2.5 rolling windows: [3, 6, 12, 24]


In [28]:
for window_hours in PM25_ROLLING_WINDOWS:
    feature_base_df[
        f"pm25_mean_{window_hours}h"
    ] = (
        feature_base_df["pm25_ug_m3"]
        .rolling(
            window=window_hours,
            min_periods=window_hours,
        )
        .mean()
    )

### Inspect the start of the rolling calculations

The first rows will contain missing rolling values because there is not enough
earlier history to complete each window.

For example:

- the 3-hour mean first becomes available on the third row
- the 6-hour mean first becomes available on the sixth row
- the 24-hour mean first becomes available after 24 hourly observations

These boundary missing values are expected.

In [29]:
PM25_ROLLING_COLUMNS = [
    "pm25_mean_3h",
    "pm25_mean_6h",
    "pm25_mean_12h",
    "pm25_mean_24h",
]

display(
    feature_base_df[
        [
            "datetime_utc",
            "pm25_ug_m3",
            *PM25_ROLLING_COLUMNS,
        ]
    ].head(30)
)

,datetime_utc,pm25_ug_m3,pm25_mean_3h,pm25_mean_6h,pm25_mean_12h,pm25_mean_24h
0,2025-07-08 00:00:00+00:00,13.202458,NaN,NaN,NaN,NaN
1,2025-07-08 01:00:00+00:00,14.268958,NaN,NaN,NaN,NaN
2,2025-07-08 02:00:00+00:00,15.587583,14.353000,NaN,NaN,NaN
3,2025-07-08 03:00:00+00:00,14.713375,14.856639,NaN,NaN,NaN
4,2025-07-08 04:00:00+00:00,17.668875,15.989944,NaN,NaN,NaN
5,2025-07-08 05:00:00+00:00,16.109417,16.163889,15.258444,NaN,NaN
6,2025-07-08 06:00:00+00:00,15.611542,16.463278,15.659958,NaN,NaN
7,2025-07-08 07:00:00+00:00,15.249333,15.656764,15.823354,NaN,NaN
8,2025-07-08 08:00:00+00:00,14.307417,15.056097,15.609993,NaN,NaN
9,2025-07-08 09:00:00+00:00,15.640125,15.065625,15.764451,NaN,NaN


In [30]:
rolling_missing_summary = (
    feature_base_df[
        PM25_ROLLING_COLUMNS
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(rolling_missing_summary)

,missing_count
pm25_mean_3h,567
pm25_mean_6h,732
pm25_mean_12h,1050
pm25_mean_24h,1634


### Verify rolling means manually

Selected rows are compared with averages calculated directly from the original
PM2.5 column.

This confirms that:

- the correct number of hourly observations is used
- the current hour is included
- no future values enter the window
- a window containing a missing value remains missing

In [31]:
validation_row_indices = [
    24,
    100,
    1000,
    5000,
]

rolling_validation_rows = []

for row_index in validation_row_indices:
    record = {
        "row_index": row_index,
        "reference_time": feature_base_df.loc[
            row_index,
            "datetime_utc",
        ],
    }

    for window_hours in PM25_ROLLING_WINDOWS:
        window_start_index = (
            row_index - window_hours + 1
        )

        source_window = feature_base_df.loc[
            window_start_index:row_index,
            "pm25_ug_m3",
        ]

        expected_value = (
            source_window.mean()
            if source_window.notna().all()
            else float("nan")
        )

        record[
            f"generated_mean_{window_hours}h"
        ] = feature_base_df.loc[
            row_index,
            f"pm25_mean_{window_hours}h",
        ]

        record[
            f"expected_mean_{window_hours}h"
        ] = expected_value

    rolling_validation_rows.append(record)


rolling_validation_df = pd.DataFrame(
    rolling_validation_rows
)

display(rolling_validation_df)

,row_index,reference_time,generated_mean_3h,expected_mean_3h,generated_mean_6h,expected_mean_6h,generated_mean_12h,expected_mean_12h,generated_mean_24h,expected_mean_24h
0,24,2025-07-09 00:00:00+00:00,16.440042,16.440042,17.047167,17.047167,19.044757,19.044757,17.423482,17.423482
1,100,2025-07-12 04:00:00+00:00,6.929472,6.929472,6.914000,6.914000,7.492549,7.492549,8.972977,8.972977
2,1000,2025-08-18 16:00:00+00:00,29.030125,29.030125,22.489472,22.489472,24.376052,24.376052,17.557031,17.557031
3,5000,2026-02-01 08:00:00+00:00,74.513722,74.513722,96.377111,96.377111,134.561948,134.561948,112.602850,112.602850


In [32]:
rolling_gap_inspection = feature_base_df.loc[
    feature_base_df["datetime_utc"].between(
        selected_gap_start
        - pd.Timedelta(hours=6),
        selected_gap_end
        + pd.Timedelta(hours=26),
    ),
    [
        "datetime_utc",
        "pm25_ug_m3",
        "pm25_mean_3h",
        "pm25_mean_6h",
        "pm25_mean_12h",
        "pm25_mean_24h",
    ],
].copy()

display(rolling_gap_inspection)

,datetime_utc,pm25_ug_m3,pm25_mean_3h,pm25_mean_6h,pm25_mean_12h,pm25_mean_24h
482,2025-07-28 02:00:00+00:00,9.551792,9.702430,9.411021,11.003115,12.081488
483,2025-07-28 03:00:00+00:00,9.688833,9.693028,9.566083,10.595371,11.677856
484,2025-07-28 04:00:00+00:00,10.953000,10.064542,9.869660,10.319319,11.442010
485,2025-07-28 05:00:00+00:00,9.424583,10.022139,9.862285,10.103830,11.289536
486,2025-07-28 06:00:00+00:00,9.438167,9.938583,9.815806,9.784538,11.163797
487,2025-07-28 07:00:00+00:00,10.185264,9.682671,9.873606,9.643321,11.015608
488,2025-07-28 08:00:00+00:00,NaN,NaN,NaN,NaN,NaN
489,2025-07-28 09:00:00+00:00,8.530167,NaN,NaN,NaN,NaN
490,2025-07-28 10:00:00+00:00,7.317125,NaN,NaN,NaN,NaN
491,2025-07-28 11:00:00+00:00,9.012250,8.286514,NaN,NaN,NaN


### PM2.5 change features

Lag and rolling features describe recent pollution levels, but they do not
directly tell the model whether PM2.5 is rising or falling.

Change features measure the difference between the current PM2.5 value and an
earlier value.

For example:

- a positive `pm25_change_1h` means PM2.5 increased during the last hour
- a negative `pm25_change_6h` means PM2.5 decreased compared with six hours ago
- a large positive `pm25_change_24h` may indicate that pollution is much higher
  than at the same time on the previous day

Only current and historical PM2.5 values are used. Missing values remain
missing.

### Selected change periods

The initial feature set uses changes over:

- 1 hour for immediate movement
- 6 hours for short-term pollution direction
- 24 hours for daily comparison

The change is calculated as:

`current PM2.5 - lagged PM2.5`

Positive values represent an increase, while negative values represent a
decrease.

In [33]:
PM25_CHANGE_HOURS = [
    1,
    6,
    24,
]

print(
    "PM2.5 change periods:",
    PM25_CHANGE_HOURS,
)

PM2.5 change periods: [1, 6, 24]


In [34]:
for change_hours in PM25_CHANGE_HOURS:
    feature_base_df[
        f"pm25_change_{change_hours}h"
    ] = (
        feature_base_df["pm25_current"]
        - feature_base_df[
            f"pm25_lag_{change_hours}h"
        ]
    )

### Inspect the generated change features

The earliest rows contain missing change values because the corresponding lag
history does not yet exist.

A change feature is also missing whenever either:

- the current PM2.5 value is missing
- the required historical PM2.5 value is missing

This is intentional because a valid difference cannot be calculated from an
incomplete pair.

In [35]:
PM25_CHANGE_COLUMNS = [
    "pm25_change_1h",
    "pm25_change_6h",
    "pm25_change_24h",
]

display(
    feature_base_df[
        [
            "datetime_utc",
            "pm25_current",
            "pm25_lag_1h",
            "pm25_change_1h",
            "pm25_lag_6h",
            "pm25_change_6h",
            "pm25_lag_24h",
            "pm25_change_24h",
        ]
    ].head(30)
)

,datetime_utc,pm25_current,pm25_lag_1h,pm25_change_1h,pm25_lag_6h,pm25_change_6h,pm25_lag_24h,pm25_change_24h
0,2025-07-08 00:00:00+00:00,13.202458,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-07-08 01:00:00+00:00,14.268958,13.202458,1.066500,NaN,NaN,NaN,NaN
2,2025-07-08 02:00:00+00:00,15.587583,14.268958,1.318625,NaN,NaN,NaN,NaN
3,2025-07-08 03:00:00+00:00,14.713375,15.587583,-0.874208,NaN,NaN,NaN,NaN
4,2025-07-08 04:00:00+00:00,17.668875,14.713375,2.955500,NaN,NaN,NaN,NaN
5,2025-07-08 05:00:00+00:00,16.109417,17.668875,-1.559458,NaN,NaN,NaN,NaN
6,2025-07-08 06:00:00+00:00,15.611542,16.109417,-0.497875,13.202458,2.409083,NaN,NaN
7,2025-07-08 07:00:00+00:00,15.249333,15.611542,-0.362208,14.268958,0.980375,NaN,NaN
8,2025-07-08 08:00:00+00:00,14.307417,15.249333,-0.941917,15.587583,-1.280166,NaN,NaN
9,2025-07-08 09:00:00+00:00,15.640125,14.307417,1.332708,14.713375,0.926750,NaN,NaN


In [36]:
change_missing_summary = (
    feature_base_df[
        PM25_CHANGE_COLUMNS
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(change_missing_summary)

,missing_count
pm25_change_1h,511
pm25_change_6h,602
pm25_change_24h,677


## **2C.** Current weather and time features

PM2.5 levels are influenced by atmospheric conditions such as temperature,
humidity, pressure, precipitation, cloud cover, and wind.

The canonical dataset already contains weather observations for every reference
hour. These values are retained as current-weather features.

Calendar features are also created because pollution patterns may vary by:

- hour of day
- day of week
- month of year

Some time variables are cyclical. For example, hour 23 and hour 0 are only one
hour apart, even though their numeric values are far apart. Sine and cosine
encodings preserve this circular relationship.

Wind direction is encoded in the same way because 359 degrees and 1 degree
represent nearly the same direction.

In [39]:
CURRENT_WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "surface_pressure",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
]

missing_current_weather_columns = [
    column
    for column in CURRENT_WEATHER_COLUMNS
    if column not in feature_base_df.columns
]

assert not missing_current_weather_columns, (
    "Missing current weather columns: "
    f"{missing_current_weather_columns}"
)

print("Current weather columns validated.")

Current weather columns validated.


In [40]:
feature_base_df["reference_hour"] = (
    feature_base_df["datetime_utc"].dt.hour
)

feature_base_df["reference_day_of_week"] = (
    feature_base_df["datetime_utc"].dt.dayofweek
)

feature_base_df["reference_month"] = (
    feature_base_df["datetime_utc"].dt.month
)

In [41]:
# Create cyclical encodings
import numpy as np

In [42]:
feature_base_df["reference_hour_sin"] = np.sin(
    2 * np.pi
    * feature_base_df["reference_hour"]
    / 24
)

feature_base_df["reference_hour_cos"] = np.cos(
    2 * np.pi
    * feature_base_df["reference_hour"]
    / 24
)

feature_base_df["reference_day_of_week_sin"] = np.sin(
    2 * np.pi
    * feature_base_df["reference_day_of_week"]
    / 7
)

feature_base_df["reference_day_of_week_cos"] = np.cos(
    2 * np.pi
    * feature_base_df["reference_day_of_week"]
    / 7
)

feature_base_df["reference_month_sin"] = np.sin(
    2 * np.pi
    * (feature_base_df["reference_month"] - 1)
    / 12
)

feature_base_df["reference_month_cos"] = np.cos(
    2 * np.pi
    * (feature_base_df["reference_month"] - 1)
    / 12
)

In [43]:
wind_direction_radians = np.deg2rad(
    feature_base_df["wind_direction_10m"]
)

feature_base_df["wind_direction_10m_sin"] = np.sin(
    wind_direction_radians
)

feature_base_df["wind_direction_10m_cos"] = np.cos(
    wind_direction_radians
)

In [44]:
REFERENCE_TIME_FEATURE_COLUMNS = [
    "reference_hour",
    "reference_day_of_week",
    "reference_month",
    "reference_hour_sin",
    "reference_hour_cos",
    "reference_day_of_week_sin",
    "reference_day_of_week_cos",
    "reference_month_sin",
    "reference_month_cos",
    "wind_direction_10m_sin",
    "wind_direction_10m_cos",
]

display(
    feature_base_df[
        [
            "datetime_utc",
            "wind_direction_10m",
            *REFERENCE_TIME_FEATURE_COLUMNS,
        ]
    ].head(24)
)

,datetime_utc,wind_direction_10m,reference_hour,reference_day_of_week,reference_month,reference_hour_sin,reference_hour_cos,reference_day_of_week_sin,reference_day_of_week_cos,reference_month_sin,reference_month_cos,wind_direction_10m_sin,wind_direction_10m_cos
0,2025-07-08 00:00:00+00:00,225,0,1,7,0.000000e+00,1.000000e+00,0.781831,0.62349,1.224647e-16,-1.0,-0.707107,-7.071068e-01
1,2025-07-08 01:00:00+00:00,263,1,1,7,2.588190e-01,9.659258e-01,0.781831,0.62349,1.224647e-16,-1.0,-0.992546,-1.218693e-01
2,2025-07-08 02:00:00+00:00,270,2,1,7,5.000000e-01,8.660254e-01,0.781831,0.62349,1.224647e-16,-1.0,-1.000000,-1.836970e-16
3,2025-07-08 03:00:00+00:00,228,3,1,7,7.071068e-01,7.071068e-01,0.781831,0.62349,1.224647e-16,-1.0,-0.743145,-6.691306e-01
4,2025-07-08 04:00:00+00:00,199,4,1,7,8.660254e-01,5.000000e-01,0.781831,0.62349,1.224647e-16,-1.0,-0.325568,-9.455186e-01
5,2025-07-08 05:00:00+00:00,198,5,1,7,9.659258e-01,2.588190e-01,0.781831,0.62349,1.224647e-16,-1.0,-0.309017,-9.510565e-01
6,2025-07-08 06:00:00+00:00,208,6,1,7,1.000000e+00,6.123234e-17,0.781831,0.62349,1.224647e-16,-1.0,-0.469472,-8.829476e-01
7,2025-07-08 07:00:00+00:00,206,7,1,7,9.659258e-01,-2.588190e-01,0.781831,0.62349,1.224647e-16,-1.0,-0.438371,-8.987940e-01
8,2025-07-08 08:00:00+00:00,201,8,1,7,8.660254e-01,-5.000000e-01,0.781831,0.62349,1.224647e-16,-1.0,-0.358368,-9.335804e-01
9,2025-07-08 09:00:00+00:00,206,9,1,7,7.071068e-01,-7.071068e-01,0.781831,0.62349,1.224647e-16,-1.0,-0.438371,-8.987940e-01


In [45]:
assert feature_base_df["reference_hour"].between(0, 23).all()

assert feature_base_df[
    "reference_day_of_week"
].between(0, 6).all()

assert feature_base_df[
    "reference_month"
].between(1, 12).all()

CYCLICAL_COLUMNS = [
    "reference_hour_sin",
    "reference_hour_cos",
    "reference_day_of_week_sin",
    "reference_day_of_week_cos",
    "reference_month_sin",
    "reference_month_cos",
    "wind_direction_10m_sin",
    "wind_direction_10m_cos",
]

assert (
    feature_base_df[CYCLICAL_COLUMNS]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    feature_base_df[CYCLICAL_COLUMNS]
    .apply(lambda column: column.between(-1, 1).all())
    .all()
)

assert (
    feature_base_df[CURRENT_WEATHER_COLUMNS]
    .isna()
    .sum()
    .sum()
    == 0
)

print(
    "Current weather, reference-time, and cyclical features validated."
)

Current weather, reference-time, and cyclical features validated.


## **2D.** Expand each reference hour into forecast horizons 1–72

The forecasting model will use a direct multi-horizon design.

Instead of predicting one hour repeatedly and feeding each prediction back into
the model, one shared regression model will learn to predict different future
horizons directly.

Each training row represents:

- one reference timestamp
- one forecast horizon from 1 to 72 hours
- one target timestamp

The forecast horizon itself becomes an input feature, allowing the same model
to learn how prediction behavior changes as the forecast distance increases.

At this stage, only the reference-to-target structure is created. Target PM2.5
and target-hour weather will be attached in the next step.

In [46]:
FORECAST_HORIZONS = list(range(1, 73))

print("First forecast horizon:", FORECAST_HORIZONS[0])
print("Last forecast horizon:", FORECAST_HORIZONS[-1])
print("Number of horizons:", len(FORECAST_HORIZONS))

First forecast horizon: 1
Last forecast horizon: 72
Number of horizons: 72


### Prepare the reference-time feature table

The current hourly feature table contains both source columns and engineered
reference-time features.

Before expansion, the UTC timestamp is renamed to `reference_time` so its role
is explicit.

Every resulting training row will preserve the same reference-time PM2.5,
weather, lag, rolling, change, and calendar features while changing only the
forecast horizon and target timestamp.

In [47]:
reference_df = feature_base_df.rename(
    columns={
        "datetime_utc": "reference_time",
    }
).copy()

print("Reference table shape:", reference_df.shape)
print(
    "Reference time range:",
    reference_df["reference_time"].min(),
    "to",
    reference_df["reference_time"].max(),
)

Reference table shape: (9144, 43)
Reference time range: 2025-07-08 00:00:00+00:00 to 2026-07-23 23:00:00+00:00


In [49]:
horizon_df = pd.DataFrame(
    {
        "forecast_horizon_hours": FORECAST_HORIZONS,
    }
)

display(horizon_df.head())
display(horizon_df.tail())

,forecast_horizon_hours
0,1
1,2
2,3
3,4
4,5


,forecast_horizon_hours
67,68
68,69
69,70
70,71
71,72


### Expand every reference timestamp across all 72 horizons

A cross join pairs every reference row with every forecast horizon.

Before removing invalid references or unavailable targets, the maximum number
of potential rows is:

`9,144 reference hours × 72 horizons = 658,368 rows`

Some rows near the end of the historical period will later be removed because
their target timestamps fall outside the available dataset.

Rows affected by missing PM2.5 history will also be removed later.

In [50]:
expanded_df = reference_df.merge(
    horizon_df,
    how="cross",
)

print("Reference rows:", len(reference_df))
print("Forecast horizons:", len(horizon_df))
print("Expanded rows:", len(expanded_df))

Reference rows: 9144
Forecast horizons: 72
Expanded rows: 658368


In [51]:
expanded_df["target_time"] = (
    expanded_df["reference_time"]
    + pd.to_timedelta(
        expanded_df["forecast_horizon_hours"],
        unit="h",
    )
)

print(
    "First target timestamp:",
    expanded_df["target_time"].min(),
)

print(
    "Last target timestamp:",
    expanded_df["target_time"].max(),
)

First target timestamp: 2025-07-08 01:00:00+00:00
Last target timestamp: 2026-07-26 23:00:00+00:00


### Inspect the 72 rows created for one reference time

The following table confirms that one reference timestamp produces exactly 72
rows and that the target timestamp moves forward by the selected number of
hours.

In [52]:
sample_reference_time = expanded_df[
    "reference_time"
].iloc[0]

sample_horizon_rows = expanded_df.loc[
    expanded_df["reference_time"].eq(
        sample_reference_time
    ),
    [
        "reference_time",
        "forecast_horizon_hours",
        "target_time",
        "pm25_current",
    ],
]

print("Sample reference time:", sample_reference_time)
print("Rows for sample reference:", len(sample_horizon_rows))

display(sample_horizon_rows.head(10))
display(sample_horizon_rows.tail(5))

Sample reference time: 2025-07-08 00:00:00+00:00
Rows for sample reference: 72


,reference_time,forecast_horizon_hours,target_time,pm25_current
0,2025-07-08 00:00:00+00:00,1,2025-07-08 01:00:00+00:00,13.202458
1,2025-07-08 00:00:00+00:00,2,2025-07-08 02:00:00+00:00,13.202458
2,2025-07-08 00:00:00+00:00,3,2025-07-08 03:00:00+00:00,13.202458
3,2025-07-08 00:00:00+00:00,4,2025-07-08 04:00:00+00:00,13.202458
4,2025-07-08 00:00:00+00:00,5,2025-07-08 05:00:00+00:00,13.202458
5,2025-07-08 00:00:00+00:00,6,2025-07-08 06:00:00+00:00,13.202458
6,2025-07-08 00:00:00+00:00,7,2025-07-08 07:00:00+00:00,13.202458
7,2025-07-08 00:00:00+00:00,8,2025-07-08 08:00:00+00:00,13.202458
8,2025-07-08 00:00:00+00:00,9,2025-07-08 09:00:00+00:00,13.202458
9,2025-07-08 00:00:00+00:00,10,2025-07-08 10:00:00+00:00,13.202458


,reference_time,forecast_horizon_hours,target_time,pm25_current
67,2025-07-08 00:00:00+00:00,68,2025-07-10 20:00:00+00:00,13.202458
68,2025-07-08 00:00:00+00:00,69,2025-07-10 21:00:00+00:00,13.202458
69,2025-07-08 00:00:00+00:00,70,2025-07-10 22:00:00+00:00,13.202458
70,2025-07-08 00:00:00+00:00,71,2025-07-10 23:00:00+00:00,13.202458
71,2025-07-08 00:00:00+00:00,72,2025-07-11 00:00:00+00:00,13.202458


In [53]:
expected_target_time = (
    expanded_df["reference_time"]
    + pd.to_timedelta(
        expanded_df["forecast_horizon_hours"],
        unit="h",
    )
)

assert expanded_df[
    "target_time"
].equals(
    expected_target_time
)

assert expanded_df[
    "forecast_horizon_hours"
].between(1, 72).all()

assert (
    expanded_df["forecast_horizon_hours"]
    .nunique()
    == 72
)

print(
    "Forecast horizon and target-time calculations validated."
)

Forecast horizon and target-time calculations validated.


In [54]:
duplicate_reference_horizon_rows = (
    expanded_df[
        [
            "reference_time",
            "forecast_horizon_hours",
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate reference-time/horizon rows:",
    duplicate_reference_horizon_rows,
)

assert duplicate_reference_horizon_rows == 0

Duplicate reference-time/horizon rows: 0


In [55]:
horizons_per_reference = (
    expanded_df
    .groupby("reference_time")
    ["forecast_horizon_hours"]
    .count()
)

print(
    "Minimum horizons per reference:",
    horizons_per_reference.min(),
)

print(
    "Maximum horizons per reference:",
    horizons_per_reference.max(),
)

assert horizons_per_reference.min() == 72
assert horizons_per_reference.max() == 72

print(
    "Each reference timestamp contains exactly 72 forecast horizons."
)

Minimum horizons per reference: 72
Maximum horizons per reference: 72
Each reference timestamp contains exactly 72 forecast horizons.


## **2E.** Attach target values and target-hour weather

Each expanded row currently identifies a reference time, forecast horizon, and
target time.

The next step attaches the values observed at the target timestamp:

- `target_pm25_ug_m3` is the value the model will learn to predict
- target-hour weather values become model inputs

For offline training, historical observed weather is used as a proxy for
forecast weather. In production, the same target-hour weather features will be
provided by the Open-Meteo forecast API.

Using observed historical weather can make offline evaluation slightly more
optimistic than real deployment because actual future weather is more accurate
than a weather forecast.

In [56]:
target_lookup_columns = [
    "datetime_utc",
    "pm25_ug_m3",
    *WEATHER_COLUMNS,
]

target_lookup_df = canonical_df[
    target_lookup_columns
].copy()

target_lookup_df = target_lookup_df.rename(
    columns={
        "datetime_utc": "target_time",
        "pm25_ug_m3": "target_pm25_ug_m3",
        **{
            column: f"target_{column}"
            for column in WEATHER_COLUMNS
        },
    }
)

print("Target lookup shape:", target_lookup_df.shape)
print("Target lookup columns:")

for column in target_lookup_df.columns:
    print(f"- {column}")

Target lookup shape: (9144, 12)
Target lookup columns:
- target_time
- target_pm25_ug_m3
- target_temperature_2m
- target_relative_humidity_2m
- target_dew_point_2m
- target_surface_pressure
- target_precipitation
- target_rain
- target_cloud_cover
- target_wind_speed_10m
- target_wind_direction_10m
- target_wind_gusts_10m


In [57]:
training_candidates_df = expanded_df.merge(
    target_lookup_df,
    on="target_time",
    how="left",
    validate="many_to_one",
)

print(
    "Rows before target attachment:",
    len(expanded_df),
)

print(
    "Rows after target attachment:",
    len(training_candidates_df),
)

Rows before target attachment: 658368
Rows after target attachment: 658368


In [58]:
TARGET_WEATHER_COLUMNS = [
    f"target_{column}"
    for column in WEATHER_COLUMNS
]

print(
    "Missing target PM2.5 values:",
    training_candidates_df[
        "target_pm25_ug_m3"
    ]
    .isna()
    .sum(),
)

print(
    "Rows missing any target weather value:",
    training_candidates_df[
        TARGET_WEATHER_COLUMNS
    ]
    .isna()
    .any(axis=1)
    .sum(),
)

Missing target PM2.5 values: 35316
Rows missing any target weather value: 2628


In [59]:
display(
    training_candidates_df[
        [
            "reference_time",
            "forecast_horizon_hours",
            "target_time",
            "target_pm25_ug_m3",
            "target_temperature_2m",
        ]
    ].tail(80)
)

,reference_time,forecast_horizon_hours,target_time,target_pm25_ug_m3,target_temperature_2m
658288,2026-07-23 22:00:00+00:00,65,2026-07-26 15:00:00+00:00,NaN,NaN
658289,2026-07-23 22:00:00+00:00,66,2026-07-26 16:00:00+00:00,NaN,NaN
658290,2026-07-23 22:00:00+00:00,67,2026-07-26 17:00:00+00:00,NaN,NaN
658291,2026-07-23 22:00:00+00:00,68,2026-07-26 18:00:00+00:00,NaN,NaN
658292,2026-07-23 22:00:00+00:00,69,2026-07-26 19:00:00+00:00,NaN,NaN
...,...,...,...,...,...
658363,2026-07-23 23:00:00+00:00,68,2026-07-26 19:00:00+00:00,NaN,NaN
658364,2026-07-23 23:00:00+00:00,69,2026-07-26 20:00:00+00:00,NaN,NaN
658365,2026-07-23 23:00:00+00:00,70,2026-07-26 21:00:00+00:00,NaN,NaN
658366,2026-07-23 23:00:00+00:00,71,2026-07-26 22:00:00+00:00,NaN,NaN


### Target-time calendar features

Pollution behavior can depend on the hour, day, and season of the timestamp
being predicted.

Target-time features describe when the forecasted PM2.5 value will occur,
rather than when the prediction is generated.

Cyclical encodings are used so that neighboring time values remain close, such
as hour 23 and hour 0.

In [60]:
training_candidates_df["target_hour"] = (
    training_candidates_df["target_time"].dt.hour
)

training_candidates_df["target_day_of_week"] = (
    training_candidates_df["target_time"].dt.dayofweek
)

training_candidates_df["target_month"] = (
    training_candidates_df["target_time"].dt.month
)


training_candidates_df["target_hour_sin"] = np.sin(
    2
    * np.pi
    * training_candidates_df["target_hour"]
    / 24
)

training_candidates_df["target_hour_cos"] = np.cos(
    2
    * np.pi
    * training_candidates_df["target_hour"]
    / 24
)

training_candidates_df[
    "target_day_of_week_sin"
] = np.sin(
    2
    * np.pi
    * training_candidates_df["target_day_of_week"]
    / 7
)

training_candidates_df[
    "target_day_of_week_cos"
] = np.cos(
    2
    * np.pi
    * training_candidates_df["target_day_of_week"]
    / 7
)

training_candidates_df["target_month_sin"] = np.sin(
    2
    * np.pi
    * (
        training_candidates_df["target_month"]
        - 1
    )
    / 12
)

training_candidates_df["target_month_cos"] = np.cos(
    2
    * np.pi
    * (
        training_candidates_df["target_month"]
        - 1
    )
    / 12
)

In [61]:
target_wind_direction_radians = np.deg2rad(
    training_candidates_df[
        "target_wind_direction_10m"
    ]
)

training_candidates_df[
    "target_wind_direction_10m_sin"
] = np.sin(
    target_wind_direction_radians
)

training_candidates_df[
    "target_wind_direction_10m_cos"
] = np.cos(
    target_wind_direction_radians
)

In [62]:
assert len(training_candidates_df) == 658_368

assert (
    training_candidates_df[
        [
            "reference_time",
            "forecast_horizon_hours",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

within_history_mask = (
    training_candidates_df["target_time"]
    <= canonical_df["datetime_utc"].max()
)

assert (
    training_candidates_df.loc[
        within_history_mask,
        TARGET_WEATHER_COLUMNS,
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    training_candidates_df.loc[
        ~within_history_mask,
        TARGET_WEATHER_COLUMNS,
    ]
    .isna()
    .all(axis=1)
    .all()
)

assert training_candidates_df[
    "forecast_horizon_hours"
].between(1, 72).all()

print(
    "Target PM2.5, target weather, and target-time features validated."
)

Target PM2.5, target weather, and target-time features validated.


## **2F.** Select valid training rows and prevent leakage

The expanded dataset contains all potential reference-time and forecast-horizon
combinations.

Not every candidate row can be used for training.

A row is valid only when:

- current PM2.5 is available
- every required PM2.5 lag is available
- every required rolling feature is available
- every PM2.5 change feature is available
- current weather is complete
- target-hour weather is complete
- target PM2.5 is available

Rows with incomplete pollution history are removed rather than filled or
interpolated.

The target PM2.5 column is used only as the prediction target. It must never be
included in the model input feature list.

In [63]:
PM25_FEATURE_COLUMNS = [
    "pm25_current",
    "pm25_lag_1h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "pm25_lag_12h",
    "pm25_lag_24h",
    "pm25_mean_3h",
    "pm25_mean_6h",
    "pm25_mean_12h",
    "pm25_mean_24h",
    "pm25_change_1h",
    "pm25_change_6h",
    "pm25_change_24h",
]

REFERENCE_TIME_COLUMNS = [
    "reference_hour",
    "reference_day_of_week",
    "reference_month",
    "reference_hour_sin",
    "reference_hour_cos",
    "reference_day_of_week_sin",
    "reference_day_of_week_cos",
    "reference_month_sin",
    "reference_month_cos",
]

CURRENT_WIND_COLUMNS = [
    "wind_direction_10m_sin",
    "wind_direction_10m_cos",
]

TARGET_TIME_COLUMNS = [
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_hour_sin",
    "target_hour_cos",
    "target_day_of_week_sin",
    "target_day_of_week_cos",
    "target_month_sin",
    "target_month_cos",
]

TARGET_WIND_COLUMNS = [
    "target_wind_direction_10m_sin",
    "target_wind_direction_10m_cos",
]

In [64]:
MODEL_FEATURE_COLUMNS = [
    *PM25_FEATURE_COLUMNS,
    *WEATHER_COLUMNS,
    *CURRENT_WIND_COLUMNS,
    *REFERENCE_TIME_COLUMNS,
    "forecast_horizon_hours",
    *TARGET_WEATHER_COLUMNS,
    *TARGET_WIND_COLUMNS,
    *TARGET_TIME_COLUMNS,
]

TARGET_COLUMN = "target_pm25_ug_m3"

print("Number of model features:", len(MODEL_FEATURE_COLUMNS))
print("Target column:", TARGET_COLUMN)

Number of model features: 56
Target column: target_pm25_ug_m3


### Measure why candidate rows are invalid

Before removing rows, separate masks are created for each missing-data reason.

The counts can overlap. For example, a row may have both a missing rolling
feature and a missing target PM2.5 value.

Therefore, these counts describe affected rows by category and should not be
added together to calculate the total number removed.

In [65]:
missing_pm25_history_mask = (
    training_candidates_df[
        PM25_FEATURE_COLUMNS
    ]
    .isna()
    .any(axis=1)
)

missing_current_weather_mask = (
    training_candidates_df[
        WEATHER_COLUMNS
    ]
    .isna()
    .any(axis=1)
)

missing_target_weather_mask = (
    training_candidates_df[
        TARGET_WEATHER_COLUMNS
    ]
    .isna()
    .any(axis=1)
)

missing_target_pm25_mask = (
    training_candidates_df[
        TARGET_COLUMN
    ]
    .isna()
)

In [66]:
invalid_reason_summary = pd.Series(
    {
        "missing_pm25_history_features": int(
            missing_pm25_history_mask.sum()
        ),
        "missing_current_weather": int(
            missing_current_weather_mask.sum()
        ),
        "missing_target_weather": int(
            missing_target_weather_mask.sum()
        ),
        "missing_target_pm25": int(
            missing_target_pm25_mask.sum()
        ),
    },
    name="affected_rows",
).to_frame()

display(invalid_reason_summary)

,affected_rows
missing_pm25_history_features,120744
missing_current_weather,0
missing_target_weather,2628
missing_target_pm25,35316


In [67]:
valid_training_row_mask = ~(
    missing_pm25_history_mask
    | missing_current_weather_mask
    | missing_target_weather_mask
    | missing_target_pm25_mask
)

print(
    "Potential training rows:",
    len(training_candidates_df),
)

print(
    "Valid training rows:",
    int(valid_training_row_mask.sum()),
)

print(
    "Removed training rows:",
    int((~valid_training_row_mask).sum()),
)

Potential training rows: 658368
Valid training rows: 522497
Removed training rows: 135871


In [68]:
model_dataset_df = (
    training_candidates_df.loc[
        valid_training_row_mask,
        [
            "reference_time",
            "target_time",
            *MODEL_FEATURE_COLUMNS,
            TARGET_COLUMN,
        ],
    ]
    .copy()
    .sort_values(
        [
            "reference_time",
            "forecast_horizon_hours",
        ]
    )
    .reset_index(drop=True)
)

print("Model-ready dataset shape:", model_dataset_df.shape)

Model-ready dataset shape: (522497, 59)


### Leakage checks

The model may use pollution history available at the reference timestamp, but
it must not receive future PM2.5 information.

Target-hour weather is allowed because the live application will receive
weather forecasts for future hours.

The checks below confirm that:

- the PM2.5 target is not in the feature list
- no future PM2.5-derived input columns exist
- target time is always later than reference time
- no duplicate reference-time and horizon combinations exist
- all selected features and targets are complete

In [69]:
assert TARGET_COLUMN not in MODEL_FEATURE_COLUMNS

future_pm25_feature_columns = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column.startswith("target_pm25")
]

assert not future_pm25_feature_columns, (
    "Future PM2.5 columns found in feature list: "
    f"{future_pm25_feature_columns}"
)

assert (
    model_dataset_df["target_time"]
    > model_dataset_df["reference_time"]
).all()

assert (
    model_dataset_df[
        [
            "reference_time",
            "forecast_horizon_hours",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    model_dataset_df[MODEL_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    model_dataset_df[TARGET_COLUMN]
    .isna()
    .sum()
    == 0
)

assert model_dataset_df[
    "forecast_horizon_hours"
].between(1, 72).all()

print("Model-ready dataset leakage and completeness checks passed.")

Model-ready dataset leakage and completeness checks passed.


In [70]:
display(
    model_dataset_df[
        [
            "reference_time",
            "forecast_horizon_hours",
            "target_time",
            "pm25_current",
            "pm25_lag_24h",
            "target_temperature_2m",
            "target_pm25_ug_m3",
        ]
    ].head(20)
)

,reference_time,forecast_horizon_hours,target_time,pm25_current,pm25_lag_24h,target_temperature_2m,target_pm25_ug_m3
0,2025-07-09 00:00:00+00:00,1,2025-07-09 01:00:00+00:00,17.257208,13.202458,29.1,16.553708
1,2025-07-09 00:00:00+00:00,2,2025-07-09 02:00:00+00:00,17.257208,13.202458,29.5,14.925958
2,2025-07-09 00:00:00+00:00,3,2025-07-09 03:00:00+00:00,17.257208,13.202458,30.4,16.269417
3,2025-07-09 00:00:00+00:00,4,2025-07-09 04:00:00+00:00,17.257208,13.202458,31.3,16.382833
4,2025-07-09 00:00:00+00:00,5,2025-07-09 05:00:00+00:00,17.257208,13.202458,32.1,16.394208
5,2025-07-09 00:00:00+00:00,6,2025-07-09 06:00:00+00:00,17.257208,13.202458,32.8,15.496583
6,2025-07-09 00:00:00+00:00,7,2025-07-09 07:00:00+00:00,17.257208,13.202458,32.9,15.325167
7,2025-07-09 00:00:00+00:00,8,2025-07-09 08:00:00+00:00,17.257208,13.202458,32.8,12.123667
8,2025-07-09 00:00:00+00:00,9,2025-07-09 09:00:00+00:00,17.257208,13.202458,32.3,13.791375
9,2025-07-09 00:00:00+00:00,10,2025-07-09 10:00:00+00:00,17.257208,13.202458,32.3,10.888333


## **2G.** Chronological train, validation, and test splits

Time-series data must not be randomly shuffled before splitting.

A random split could allow rows from later dates to enter the training set
while earlier dates appear in validation or testing. That would create an
unrealistic evaluation because the model would effectively learn from the
future.

The dataset is therefore divided chronologically using unique reference
timestamps:

- earliest 70% for training
- next 15% for validation
- latest 15% for testing

All forecast horizons created from the same reference timestamp remain in the
same split.

In [71]:
unique_reference_times = (
    model_dataset_df["reference_time"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(
    "Unique valid reference timestamps:",
    len(unique_reference_times),
)

print(
    "First valid reference timestamp:",
    unique_reference_times.iloc[0],
)

print(
    "Last valid reference timestamp:",
    unique_reference_times.iloc[-1],
)

Unique valid reference timestamps: 7465
First valid reference timestamp: 2025-07-09 00:00:00+00:00
Last valid reference timestamp: 2026-07-23 22:00:00+00:00


### Calculate chronological split boundaries

The split boundaries are calculated from unique reference timestamps rather
than expanded training rows.

This prevents timestamps with many valid horizons from influencing the split
position differently from timestamps with fewer valid horizons.

In [72]:
number_of_reference_times = len(
    unique_reference_times
)

train_reference_count = int(
    number_of_reference_times * 0.70
)

validation_reference_count = int(
    number_of_reference_times * 0.15
)

test_reference_count = (
    number_of_reference_times
    - train_reference_count
    - validation_reference_count
)


print(
    "Training reference timestamps:",
    train_reference_count,
)

print(
    "Validation reference timestamps:",
    validation_reference_count,
)

print(
    "Testing reference timestamps:",
    test_reference_count,
)

print(
    "Total assigned reference timestamps:",
    (
        train_reference_count
        + validation_reference_count
        + test_reference_count
    ),
)

Training reference timestamps: 5225
Validation reference timestamps: 1119
Testing reference timestamps: 1121
Total assigned reference timestamps: 7465


In [73]:
train_reference_times = set(
    unique_reference_times.iloc[
        :train_reference_count
    ]
)

validation_reference_times = set(
    unique_reference_times.iloc[
        train_reference_count:
        train_reference_count
        + validation_reference_count
    ]
)

test_reference_times = set(
    unique_reference_times.iloc[
        train_reference_count
        + validation_reference_count:
    ]
)

In [74]:
train_df = (
    model_dataset_df.loc[
        model_dataset_df["reference_time"].isin(
            train_reference_times
        )
    ]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    model_dataset_df.loc[
        model_dataset_df["reference_time"].isin(
            validation_reference_times
        )
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    model_dataset_df.loc[
        model_dataset_df["reference_time"].isin(
            test_reference_times
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows:", len(test_df))

Training rows: 369982
Validation rows: 75702
Testing rows: 76813


### Inspect the chronological ranges

The latest training reference timestamp must occur before the earliest
validation timestamp.

Similarly, the latest validation timestamp must occur before the earliest
testing timestamp.

This confirms that the three datasets do not overlap in time.

In [75]:
split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "unique_reference_times": (
                train_df["reference_time"].nunique()
            ),
            "reference_start": (
                train_df["reference_time"].min()
            ),
            "reference_end": (
                train_df["reference_time"].max()
            ),
            "target_start": (
                train_df["target_time"].min()
            ),
            "target_end": (
                train_df["target_time"].max()
            ),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "unique_reference_times": (
                validation_df[
                    "reference_time"
                ].nunique()
            ),
            "reference_start": (
                validation_df[
                    "reference_time"
                ].min()
            ),
            "reference_end": (
                validation_df[
                    "reference_time"
                ].max()
            ),
            "target_start": (
                validation_df[
                    "target_time"
                ].min()
            ),
            "target_end": (
                validation_df[
                    "target_time"
                ].max()
            ),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "unique_reference_times": (
                test_df["reference_time"].nunique()
            ),
            "reference_start": (
                test_df["reference_time"].min()
            ),
            "reference_end": (
                test_df["reference_time"].max()
            ),
            "target_start": (
                test_df["target_time"].min()
            ),
            "target_end": (
                test_df["target_time"].max()
            ),
        },
    ]
)

display(split_summary)

,split,rows,unique_reference_times,reference_start,reference_end,target_start,target_end
0,train,369982,5225,2025-07-09 00:00:00+00:00,2026-03-21 11:00:00+00:00,2025-07-09 01:00:00+00:00,2026-03-24 11:00:00+00:00
1,validation,75702,1119,2026-03-21 12:00:00+00:00,2026-05-30 21:00:00+00:00,2026-03-21 13:00:00+00:00,2026-06-02 21:00:00+00:00
2,test,76813,1121,2026-05-30 22:00:00+00:00,2026-07-23 22:00:00+00:00,2026-05-30 23:00:00+00:00,2026-07-23 23:00:00+00:00


In [76]:
assert (
    train_df["reference_time"].max()
    < validation_df["reference_time"].min()
)

assert (
    validation_df["reference_time"].max()
    < test_df["reference_time"].min()
)

print(
    "Chronological reference-time ordering validated."
)

Chronological reference-time ordering validated.


In [77]:
train_references = set(
    train_df["reference_time"].unique()
)

validation_references = set(
    validation_df["reference_time"].unique()
)

test_references = set(
    test_df["reference_time"].unique()
)


assert train_references.isdisjoint(
    validation_references
)

assert train_references.isdisjoint(
    test_references
)

assert validation_references.isdisjoint(
    test_references
)


print(
    "No reference timestamp appears in more than one split."
)

No reference timestamp appears in more than one split.


In [78]:
total_split_rows = (
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

print(
    "Original model-ready rows:",
    len(model_dataset_df),
)

print(
    "Rows across all splits:",
    total_split_rows,
)


assert total_split_rows == len(
    model_dataset_df
)

print(
    "All model-ready rows were assigned to exactly one split."
)

Original model-ready rows: 522497
Rows across all splits: 522497
All model-ready rows were assigned to exactly one split.


In [79]:
horizon_coverage_summary = pd.DataFrame(
    {
        "train_rows": (
            train_df[
                "forecast_horizon_hours"
            ]
            .value_counts()
            .sort_index()
        ),
        "validation_rows": (
            validation_df[
                "forecast_horizon_hours"
            ]
            .value_counts()
            .sort_index()
        ),
        "test_rows": (
            test_df[
                "forecast_horizon_hours"
            ]
            .value_counts()
            .sort_index()
        ),
    }
)

display(horizon_coverage_summary)

,train_rows,validation_rows,test_rows
forecast_horizon_hours,,,
1,5199,1111,1115
2,5187,1107,1110
3,5179,1104,1106
4,5174,1102,1103
5,5171,1100,1100
...,...,...,...
68,5123,1014,1036
69,5123,1013,1035
70,5123,1012,1034


In [80]:
expected_horizons = set(range(1, 73))

assert set(
    train_df[
        "forecast_horizon_hours"
    ].unique()
) == expected_horizons

assert set(
    validation_df[
        "forecast_horizon_hours"
    ].unique()
) == expected_horizons

assert set(
    test_df[
        "forecast_horizon_hours"
    ].unique()
) == expected_horizons


print(
    "All three splits contain forecast horizons 1 through 72."
)

All three splits contain forecast horizons 1 through 72.


### Prevent target overlap between dataset splits

Splitting only by reference timestamp keeps all horizons from one reference
hour together, but long forecast horizons can cross a split boundary.

For example, a training reference timestamp near the end of the training
period may have a 72-hour target that falls inside the validation period.

To create a stricter evaluation, a 72-hour purge gap is added between:

- training and validation
- validation and testing

Reference timestamps inside these boundary gaps are excluded.

This ensures that the latest target timestamp from an earlier split occurs
before the first reference timestamp of the following split.

In [81]:
MAX_FORECAST_HORIZON_HOURS = 72

PURGE_GAP = pd.Timedelta(
    hours=MAX_FORECAST_HORIZON_HOURS
)

print("Purge gap:", PURGE_GAP)

Purge gap: 3 days 00:00:00


In [82]:
validation_reference_start = (
    validation_df["reference_time"].min()
)

test_reference_start = (
    test_df["reference_time"].min()
)

print(
    "Original validation reference start:",
    validation_reference_start,
)

print(
    "Original test reference start:",
    test_reference_start,
)

Original validation reference start: 2026-03-21 12:00:00+00:00
Original test reference start: 2026-05-30 22:00:00+00:00


In [83]:
train_purged_df = (
    train_df.loc[
        train_df["target_time"]
        < validation_reference_start
    ]
    .copy()
    .reset_index(drop=True)
)

validation_purged_df = (
    validation_df.loc[
        validation_df["target_time"]
        < test_reference_start
    ]
    .copy()
    .reset_index(drop=True)
)

test_purged_df = (
    test_df
    .copy()
    .reset_index(drop=True)
)

In [84]:
train_reference_max_targets = (
    train_df
    .groupby("reference_time")["target_time"]
    .max()
)

valid_train_references = set(
    train_reference_max_targets.loc[
        train_reference_max_targets
        < validation_reference_start
    ].index
)


validation_reference_max_targets = (
    validation_df
    .groupby("reference_time")["target_time"]
    .max()
)

valid_validation_references = set(
    validation_reference_max_targets.loc[
        validation_reference_max_targets
        < test_reference_start
    ].index
)


train_purged_df = (
    train_df.loc[
        train_df["reference_time"].isin(
            valid_train_references
        )
    ]
    .copy()
    .reset_index(drop=True)
)

validation_purged_df = (
    validation_df.loc[
        validation_df["reference_time"].isin(
            valid_validation_references
        )
    ]
    .copy()
    .reset_index(drop=True)
)

test_purged_df = (
    test_df
    .copy()
    .reset_index(drop=True)
)

In [85]:
print(
    "Training rows before purge:",
    len(train_df),
)

print(
    "Training rows after purge:",
    len(train_purged_df),
)

print(
    "Validation rows before purge:",
    len(validation_df),
)

print(
    "Validation rows after purge:",
    len(validation_purged_df),
)

print(
    "Testing rows:",
    len(test_purged_df),
)

Training rows before purge: 369982
Training rows after purge: 364798
Validation rows before purge: 75702
Validation rows after purge: 71256
Testing rows: 76813


In [86]:
print(
    "Latest training target:",
    train_purged_df["target_time"].max(),
)

print(
    "First validation reference:",
    validation_purged_df[
        "reference_time"
    ].min(),
)

print(
    "Latest validation target:",
    validation_purged_df[
        "target_time"
    ].max(),
)

print(
    "First testing reference:",
    test_purged_df[
        "reference_time"
    ].min(),
)

Latest training target: 2026-03-21 11:00:00+00:00
First validation reference: 2026-03-21 12:00:00+00:00
Latest validation target: 2026-05-30 21:00:00+00:00
First testing reference: 2026-05-30 22:00:00+00:00


In [87]:
train_df = train_purged_df
validation_df = validation_purged_df
test_df = test_purged_df

## **2H.** Save the model-ready datasets

The feature-engineering pipeline has produced:

- one complete model-ready dataset
- a chronologically separated training dataset
- a validation dataset
- a testing dataset

The split datasets use a 72-hour purge rule so that targets from an earlier
split do not cross into the reference period of the following split.

Parquet is used because it preserves:

- timezone-aware timestamps
- numeric data types
- Boolean values
- column structure

It is also smaller and faster to load than CSV for model training.

In [88]:
TRAINING_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "training"
)

TRAINING_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FULL_DATASET_PATH = (
    TRAINING_DATA_DIR
    / "feature_dataset_full.parquet"
)

TRAIN_DATASET_PATH = (
    TRAINING_DATA_DIR
    / "train_dataset.parquet"
)

VALIDATION_DATASET_PATH = (
    TRAINING_DATA_DIR
    / "validation_dataset.parquet"
)

TEST_DATASET_PATH = (
    TRAINING_DATA_DIR
    / "test_dataset.parquet"
)


print("Training data directory:", TRAINING_DATA_DIR)

Training data directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training


In [89]:
final_model_dataset_df = (
    pd.concat(
        [
            train_df,
            validation_df,
            test_df,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "reference_time",
            "forecast_horizon_hours",
        ]
    )
    .reset_index(drop=True)
)


print(
    "Original model-ready rows before purge:",
    len(model_dataset_df),
)

print(
    "Final model-ready rows after purge:",
    len(final_model_dataset_df),
)

print(
    "Rows removed by split purging:",
    len(model_dataset_df)
    - len(final_model_dataset_df),
)

Original model-ready rows before purge: 522497
Final model-ready rows after purge: 512867
Rows removed by split purging: 9630


In [90]:
assert len(final_model_dataset_df) == (
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

assert (
    final_model_dataset_df[
        [
            "reference_time",
            "forecast_horizon_hours",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    final_model_dataset_df[
        MODEL_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    final_model_dataset_df[
        TARGET_COLUMN
    ]
    .isna()
    .sum()
    == 0
)

print(
    "Final post-purge model dataset validated."
)

Final post-purge model dataset validated.


### Save the datasets

The complete post-purge dataset and each chronological split are stored
separately.

The full dataset is useful for inspection and reproducibility.

The split datasets should be used directly during baseline model training so
that the chronological boundaries are not recalculated differently in later
notebooks.

In [91]:
final_model_dataset_df.to_parquet(
    FULL_DATASET_PATH,
    index=False,
)

train_df.to_parquet(
    TRAIN_DATASET_PATH,
    index=False,
)

validation_df.to_parquet(
    VALIDATION_DATASET_PATH,
    index=False,
)

test_df.to_parquet(
    TEST_DATASET_PATH,
    index=False,
)


print("Saved full dataset:", FULL_DATASET_PATH)
print("Saved training dataset:", TRAIN_DATASET_PATH)
print("Saved validation dataset:", VALIDATION_DATASET_PATH)
print("Saved testing dataset:", TEST_DATASET_PATH)

Saved full dataset: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training/feature_dataset_full.parquet
Saved training dataset: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training/train_dataset.parquet
Saved validation dataset: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training/validation_dataset.parquet
Saved testing dataset: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training/test_dataset.parquet


In [92]:
saved_dataset_paths = {
    "full": FULL_DATASET_PATH,
    "train": TRAIN_DATASET_PATH,
    "validation": VALIDATION_DATASET_PATH,
    "test": TEST_DATASET_PATH,
}

saved_file_summary = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "exists": dataset_path.exists(),
            "size_bytes": (
                dataset_path.stat().st_size
                if dataset_path.exists()
                else None
            ),
        }
        for dataset_name, dataset_path
        in saved_dataset_paths.items()
    ]
)

display(saved_file_summary)

,dataset,exists,size_bytes
0,full,True,3344313
1,train,True,2451667
2,validation,True,438555
3,test,True,454151


### Reload the saved files

Saving a file successfully does not automatically prove that it can be loaded
without schema or datatype changes.

Each Parquet file is therefore reloaded and compared with its in-memory
DataFrame.

In [93]:
full_reloaded_df = pd.read_parquet(
    FULL_DATASET_PATH
)

train_reloaded_df = pd.read_parquet(
    TRAIN_DATASET_PATH
)

validation_reloaded_df = pd.read_parquet(
    VALIDATION_DATASET_PATH
)

test_reloaded_df = pd.read_parquet(
    TEST_DATASET_PATH
)


print("Reloaded full shape:", full_reloaded_df.shape)
print("Reloaded train shape:", train_reloaded_df.shape)
print(
    "Reloaded validation shape:",
    validation_reloaded_df.shape,
)
print("Reloaded test shape:", test_reloaded_df.shape)

Reloaded full shape: (512867, 59)
Reloaded train shape: (364798, 59)
Reloaded validation shape: (71256, 59)
Reloaded test shape: (76813, 59)


In [94]:
pd.testing.assert_frame_equal(
    final_model_dataset_df,
    full_reloaded_df,
    check_dtype=True,
)

pd.testing.assert_frame_equal(
    train_df,
    train_reloaded_df,
    check_dtype=True,
)

pd.testing.assert_frame_equal(
    validation_df,
    validation_reloaded_df,
    check_dtype=True,
)

pd.testing.assert_frame_equal(
    test_df,
    test_reloaded_df,
    check_dtype=True,
)


print(
    "All saved Parquet datasets passed reload validation."
)

All saved Parquet datasets passed reload validation.


### Save feature metadata and the Phase 2 validation report

The Parquet files contain the actual model-ready data, but later training code
also needs a reliable record of:

- which columns are model inputs
- which column is the prediction target
- how many rows were generated and removed
- how the chronological splits were created
- which leakage checks passed

These details are saved as JSON so that later notebooks and application code do
not need to redefine the feature schema manually.

In [95]:
import json

In [96]:
FEATURE_COLUMNS_PATH = (
    TRAINING_DATA_DIR
    / "feature_columns.json"
)

feature_metadata = {
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "target_column": TARGET_COLUMN,
    "identifier_columns": [
        "reference_time",
        "target_time",
    ],
    "number_of_features": len(MODEL_FEATURE_COLUMNS),
    "forecast_horizon_min": 1,
    "forecast_horizon_max": 72,
}

with open(
    FEATURE_COLUMNS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_metadata,
        file,
        indent=2,
    )

print("Saved feature metadata:", FEATURE_COLUMNS_PATH)
print("Number of model features:", len(MODEL_FEATURE_COLUMNS))

Saved feature metadata: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training/feature_columns.json
Number of model features: 56


In [97]:
final_split_summary = {
    "train": {
        "rows": len(train_df),
        "unique_reference_times": int(
            train_df["reference_time"].nunique()
        ),
        "reference_start": (
            train_df["reference_time"].min().isoformat()
        ),
        "reference_end": (
            train_df["reference_time"].max().isoformat()
        ),
        "target_start": (
            train_df["target_time"].min().isoformat()
        ),
        "target_end": (
            train_df["target_time"].max().isoformat()
        ),
    },
    "validation": {
        "rows": len(validation_df),
        "unique_reference_times": int(
            validation_df["reference_time"].nunique()
        ),
        "reference_start": (
            validation_df["reference_time"].min().isoformat()
        ),
        "reference_end": (
            validation_df["reference_time"].max().isoformat()
        ),
        "target_start": (
            validation_df["target_time"].min().isoformat()
        ),
        "target_end": (
            validation_df["target_time"].max().isoformat()
        ),
    },
    "test": {
        "rows": len(test_df),
        "unique_reference_times": int(
            test_df["reference_time"].nunique()
        ),
        "reference_start": (
            test_df["reference_time"].min().isoformat()
        ),
        "reference_end": (
            test_df["reference_time"].max().isoformat()
        ),
        "target_start": (
            test_df["target_time"].min().isoformat()
        ),
        "target_end": (
            test_df["target_time"].max().isoformat()
        ),
    },
}

In [98]:
PHASE_2_REPORT_PATH = (
    TRAINING_DATA_DIR
    / "phase_2_validation_report.json"
)

phase_2_validation_report = {
    "phase": "Phase 2 - Feature engineering and training dataset generation",
    "canonical_input_rows": len(canonical_df),
    "potential_expanded_rows": len(expanded_df),
    "valid_rows_before_split_purge": len(model_dataset_df),
    "final_rows_after_split_purge": len(final_model_dataset_df),
    "rows_removed_before_split_purge": int(
        len(training_candidates_df)
        - len(model_dataset_df)
    ),
    "rows_removed_by_split_purge": int(
        len(model_dataset_df)
        - len(final_model_dataset_df)
    ),
    "invalid_row_reasons": {
        key: int(value)
        for key, value in (
            invalid_reason_summary["affected_rows"]
            .to_dict()
            .items()
        )
    },
    "feature_count": len(MODEL_FEATURE_COLUMNS),
    "target_column": TARGET_COLUMN,
    "missing_feature_values": int(
        final_model_dataset_df[
            MODEL_FEATURE_COLUMNS
        ]
        .isna()
        .sum()
        .sum()
    ),
    "missing_target_values": int(
        final_model_dataset_df[
            TARGET_COLUMN
        ]
        .isna()
        .sum()
    ),
    "duplicate_reference_horizon_keys": int(
        final_model_dataset_df[
            [
                "reference_time",
                "forecast_horizon_hours",
            ]
        ]
        .duplicated()
        .sum()
    ),
    "forecast_horizons": {
        "minimum": int(
            final_model_dataset_df[
                "forecast_horizon_hours"
            ].min()
        ),
        "maximum": int(
            final_model_dataset_df[
                "forecast_horizon_hours"
            ].max()
        ),
        "unique_count": int(
            final_model_dataset_df[
                "forecast_horizon_hours"
            ].nunique()
        ),
    },
    "split_strategy": {
        "type": "chronological_reference_time_split",
        "train_fraction": 0.70,
        "validation_fraction": 0.15,
        "test_fraction": 0.15,
        "purge_rule": (
            "Remove reference groups whose target timestamps "
            "cross into the next split."
        ),
        "maximum_forecast_horizon_hours": 72,
    },
    "splits": final_split_summary,
    "leakage_checks": {
        "target_not_in_feature_list": (
            TARGET_COLUMN not in MODEL_FEATURE_COLUMNS
        ),
        "no_future_pm25_features": (
            len(future_pm25_feature_columns) == 0
        ),
        "target_after_reference": bool(
            (
                final_model_dataset_df["target_time"]
                > final_model_dataset_df["reference_time"]
            ).all()
        ),
        "train_target_before_validation_reference": bool(
            train_df["target_time"].max()
            < validation_df["reference_time"].min()
        ),
        "validation_target_before_test_reference": bool(
            validation_df["target_time"].max()
            < test_df["reference_time"].min()
        ),
    },
}

In [99]:
with open(
    PHASE_2_REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        phase_2_validation_report,
        file,
        indent=2,
    )

print(
    "Saved Phase 2 validation report:",
    PHASE_2_REPORT_PATH,
)

Saved Phase 2 validation report: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training/phase_2_validation_report.json


In [100]:
with open(
    FEATURE_COLUMNS_PATH,
    "r",
    encoding="utf-8",
) as file:
    feature_metadata_reloaded = json.load(file)

with open(
    PHASE_2_REPORT_PATH,
    "r",
    encoding="utf-8",
) as file:
    phase_2_report_reloaded = json.load(file)


assert (
    feature_metadata_reloaded["feature_columns"]
    == MODEL_FEATURE_COLUMNS
)

assert (
    feature_metadata_reloaded["target_column"]
    == TARGET_COLUMN
)

assert (
    phase_2_report_reloaded[
        "final_rows_after_split_purge"
    ]
    == len(final_model_dataset_df)
)

assert all(
    phase_2_report_reloaded[
        "leakage_checks"
    ].values()
)

print("Feature metadata and Phase 2 report validation passed.")

Feature metadata and Phase 2 report validation passed.


## Phase 2 conclusion

The canonical hourly dataset has been transformed into a direct
multi-horizon model-training dataset.

Completed work includes:

- PM2.5 current, lag, rolling, and change features
- current weather features
- reference-time cyclical features
- forecast horizons from 1 through 72 hours
- target PM2.5 attachment
- target-hour weather attachment
- target-time cyclical features
- removal of incomplete rows
- leakage checks
- chronological train, validation, and test splits
- purge rules preventing target overlap across split boundaries
- Parquet dataset export
- feature metadata and validation report export

The next phase will train and compare baseline regression models using the
saved chronological datasets.